In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")

In [ ]:
import hashlib
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import glob
import requests

ZENODO_RECORD_ID = '7119040'

In [ ]:
def compute_md5(file_path):
    """
    Computes MD5 hash in 4KB chunks (Memory Safe).
    """
    hash_md5 = hashlib.md5()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

In [ ]:
def get_zenodo_hashes(record_id):
    """
    Fetches the official file list and MD5 checksums from Zenodo API.
    """
    api_url = f"https://zenodo.org/api/records/{record_id}"
    print(f"Fetching metadata from: {api_url}")
    
    try:
        r = requests.get(api_url)
        if r.status_code != 200:
            print(f"Error: Failed to fetch record (Status {r.status_code})")
            return None
            
        data = r.json()
        files = data.get('files', [])
        
        manifest = {}
        for f in files:
            algo, value = f['checksum'].split(':')
            if algo == 'md5':
                manifest[f['key']] = value
                
        print(f"Successfully retrieved {len(manifest)} official hashes from Zenodo.")
        return manifest
        
    except Exception as e:
        print(f"API Error: {e}")
        return None

# Fetch the official list
official_hashes = get_zenodo_hashes(ZENODO_RECORD_ID)

In [ ]:
search_pattern = str(JAMSHIELD_RAW_DIR / "*.mat") # Adjust pattern if needed
local_files = sorted(list(glob.glob(search_pattern)))

results = []

print(f"Verifying {len(local_files)} local files against Zenodo records...")
print("="*80)

for file_path in tqdm(local_files, desc="Verifying"):
    p = Path(file_path)
    filename = p.name
    
    # Calculate Local Hash
    local_hash = compute_md5(p)
    
    # Compare with Official
    expected = official_hashes.get(filename) if official_hashes else None
    
    status = "UNKNOWN"
    if expected:
        if local_hash == expected:
            status = "PASS"
        else:
            status = "CORRUPTED"
    else:
        status = "UNTRACKED" # File exists locally but not in Zenodo record

    results.append({
        'filename': filename,
        'status': status,
        'local_md5': local_hash,
        'zenodo_md5': expected
    })

# --- Convert to DataFrame ---
df_verify = pd.DataFrame(results)

# Show Failures
failures = df_verify[df_verify['status'] == 'CORRUPTED']
if not failures.empty:
    print(f"\nCRITICAL WARNING: {len(failures)} files are corrupted!")
    print(failures[['filename', 'local_md5', 'zenodo_md5']])
else:
    print("\nSUCCESS: All tracked files matched Zenodo hashes.")
